In [ ]:
# ===============================================

# In this file the architectures of belwo models are presented according to the R-GAN paper.
# The architectures of Model A, Model B and Model C
# The architectures of ResNet-18, ResNet-32 and Wide ResNet-34
# The Model AA, Model BB and Model CC  

# ===============================================

In [ ]:

# ===============================================
# This cell is the Architecture of Model A.
# ===============================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class ModelA(nn.Module):
    def __init__(self):
        super(ModelA, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, stride=1, padding=2)

        # Dropout layers
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Convolutional block
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.dropout1(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected block
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        # Return logits (no softmax for CrossEntropyLoss)
        return x


In [ ]:
# ===============================================
# This cell is the Architecture of Model B.
# ===============================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class ModelB(nn.Module):
    def __init__(self):
        super(ModelB, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 64, kernel_size=8, stride=1, padding=0)
        self.dropout1 = nn.Dropout(0.2)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=6, stride=1, padding=0)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=5, stride=1, padding=0)
        self.dropout2 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 12 * 12, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.dropout1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.dropout2(x)

        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x  # logits (Softmax handled externally during evaluation)


In [ ]:
# ===============================================
# This cell is the Architecture of Model C.
# ===============================================


class ModelC(nn.Module):
    def __init__(self):
        super(ModelC, self).__init__()
        # --- Convolutional layers ---
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)  # Conv(32,3,3)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)                           # Conv(32,3,3)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)                           # Conv(64,3,3)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)                                 # MaxPooling(2,2)

        self.fc1 = nn.Linear(64 * 8 * 8, 200)                                              # FC(200)
        self.fc2 = nn.Linear(200, 10)                                                      # FC(10)

    def forward(self, x):
        # --- Block 1 ---
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)

        # --- Block 2 ---
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        # --- Flatten and FC ---
        x = x.view(x.size(0), -1)      # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=1)        # Softmax for probabilities
        return x


In [ ]:
# ===============================================
# This cell is the Architecture of Model ResNet-18 (CIFAR version).
# ===============================================


# Basic residual block (same as your provided snippet)
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ResNet-18 adapted for CIFAR-10 (your provided)
class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_CIFAR, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)  # global avg pool
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18_CIFAR():
    return ResNet_CIFAR(BasicBlock, [2,2,2,2])




In [ ]:
# ===============================================
# This cell is the Architecture of Model ResNet-32 (CIFAR version).
# ===============================================


class BasicBlock_CIFAR(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock_CIFAR, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet32_CIFAR(nn.Module):
    """
    ResNet-32 for CIFAR-10: (3 + 5 + 5 + 5) blocks, total depth 32.
    """
    def __init__(self, block=BasicBlock_CIFAR, num_blocks=[5, 5, 5], num_classes=10):
        super(ResNet32_CIFAR, self).__init__()
        self.in_planes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet32_CIFAR10():
    """Convenience constructor"""
    return ResNet32_CIFAR()


In [ ]:
# ===============================================
# This cell is the Architecture of Model Wide ReNet-34 (CIFAR version).
# ===============================================


class BasicBlockWide(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1, dropRate=0.0):
        super(BasicBlockWide, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.shortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False) or None

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.shortcut is None else self.shortcut(x), out)


class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        layers = []
        for i in range(nb_layers):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes,
                                i == 0 and stride or 1, dropRate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet_CIFAR(nn.Module):
    def __init__(self, depth=34, widen_factor=10, num_classes=10, dropRate=0.0):
        super(WideResNet_CIFAR, self).__init__()
        nChannels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert ((depth - 4) % 6 == 0), 'Depth should be 6n+4'
        n = (depth - 4) // 6
        block = BasicBlockWide

        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(nChannels[3], num_classes)

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        out = out.view(out.size(0), -1)
        return self.fc(out)


def WideResNet34_10_CIFAR():
    """Return WRN-34-10 (depth=34, widen_factor=10)"""
    return WideResNet_CIFAR(depth=34, widen_factor=10, num_classes=10)


In [ ]:
# ===============================================
# This cell is the Architecture of Model AA.
# ===============================================

class ModelAA(nn.Module):
    def __init__(self):
        super(ModelAA, self).__init__()

        self.conv1 = nn.Conv2d(3, 64, kernel_size=5, stride=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, stride=2)

        self.dropout1 = nn.Dropout(0.25)

        # The input size to fc1 is 64 channels * 12x12 feature map size (for 32x32 input)
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.dropout1(x)

        x = x.view(x.size(0), -1) # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x) # Logits

        return x


In [ ]:

# ===============================================
# This cell is the Architecture of Model BB.
# ===============================================


class ModelBB(nn.Module):
    def __init__(self):
        super(ModelBB, self).__init__()

        # Dropout layer first, applied in forward
        self.dropout_input = nn.Dropout(0.2)

        # Conv1: 3->64, 8x8, stride 2.
        # (32-8)/2 + 1 = 13x13 output
        self.conv1 = nn.Conv2d(3, 64, kernel_size=8, stride=2)

        # Conv2: 64->128, 6x6, stride 2.
        self.conv2 = nn.Conv2d(64, 128, kernel_size=6, stride=2)

        # Conv3: 128->128, 5x5, stride 1.
        self.conv3 = nn.Conv2d(128, 128, kernel_size=5, stride=1, padding=2)

        # Final Feature Map Size: 128 channels * 4 * 4 spatial size = 2048
        self.dropout_fc = nn.Dropout(0.5)
        self.fc = nn.Linear(128 * 4 * 4, 10)

    def forward(self, x):
        # Dropout(0.2) on input
        x = self.dropout_input(x)

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.view(x.size(0), -1)
        # Dropout(0.5)
        x = self.dropout_fc(x)
        # FC(10)
        x = self.fc(x)

        return x

In [ ]:

# ===============================================
# This cell is the Architecture of Model CC.
# ===============================================


class ModelCC(nn.Module):
    def __init__(self):
        super(ModelCC, self).__init__()

        # Conv1: 3->128, 3x3, stride 1.
        self.conv1 = nn.Conv2d(3, 128, kernel_size=3, stride=1)

        # Conv2: 128->64, 3x3, stride 2.
        self.conv2 = nn.Conv2d(128, 64, kernel_size=3, stride=2)

        # Final Feature Map Size: 64 channels * 14 * 14 spatial size = 12544
        self.dropout1 = nn.Dropout(0.25)

        # FC(128)
        self.fc1 = nn.Linear(64 * 14 * 14, 128)
        self.dropout2 = nn.Dropout(0.5)

        # FC(10)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Dropout(0.25)
        x = self.dropout1(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        # Dropout(0.5)
        x = self.dropout2(x)
        # FC(10)
        x = self.fc2(x)

        return x